In [0]:
spark.table("default.gold_suburb_metrics").show(10)
spark.table("default.gold_property_count").show(10)
spark.table("default.gold_property_type").show()

+------------+------------------+
|      Suburb|         avg_price|
+------------+------------------+
|     Kooyong|         2185000.0|
|  Canterbury|2180240.7407407407|
| Middle Park|2082529.4117647058|
| Albert Park| 1941355.072463768|
|    Brighton|         1930158.0|
|      Balwyn|1869878.5046728973|
|   Eaglemont| 1831695.652173913|
|Balwyn North|1793405.2631578948|
|     Malvern| 1764992.537313433|
|         Kew|1758435.0282485876|
+------------+------------------+
only showing top 10 rows
+--------------+--------------+
|        Suburb|property_count|
+--------------+--------------+
|   Albert Park|            69|
|Brunswick West|           110|
|        Elwood|           131|
|     Maidstone|            94|
|   Mont Albert|            34|
|       Niddrie|            82|
|        Ormond|            83|
|West Footscray|           105|
| Taylors Lakes|            22|
|      Mulgrave|            29|
+--------------+--------------+
only showing top 10 rows
+----+-----------------+
|

In [0]:
spark.sql("SHOW TABLES").show()

+--------+-------------------+-----------+
|database|          tableName|isTemporary|
+--------+-------------------+-----------+
| default|bronze_melb_housing|      false|
| default|gold_property_count|      false|
| default| gold_property_type|      false|
| default|gold_suburb_metrics|      false|
| default|          melb_data|      false|
| default|silver_melb_housing|      false|
+--------+-------------------+-----------+



In [0]:
%python
from pyspark.sql.functions import avg, count
silver_df=spark.table("default.silver_melb_housing")

gold_suburb_metrics = (
    silver_df
        .groupBy("Suburb")
        .agg(avg("Price").alias("avg_price"))
        .orderBy("avg_price", ascending=False)
)
# from pyspark.sql.functions import count

gold_property_count = (
    silver_df
        .groupBy("Suburb")
        .agg(count("*").alias("property_count"))
)
gold_property_type = (
    silver_df
        .groupBy("Type")
        .agg(avg("Price").alias("avg_price"))
)
# Save
gold_suburb_metrics.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("default.gold_suburb_metrics")
gold_property_count.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("default.gold_property_count") 
gold_property_type.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("default.gold_property_type" )